In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))

from pathlib import Path
import torch as th
import numpy as np
from imitation.util import logger as imit_logger
import datetime
from stable_baselines3.common.vec_env import SubprocVecEnv
from mario import make_env
from sb3_contrib import RecurrentPPO

import gc
from IPython import get_ipython

from utils import get_last_index

current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"./tensorboard_ppo/run_{current_time}"
custom_logger = imit_logger.configure(log_path, ["tensorboard", "stdout"])

gc.collect()
th.cuda.empty_cache()

train_path = './recurrent_ppo/'
sub_train_path = train_path + "steps/"

os.makedirs(train_path, exist_ok=True)
os.makedirs(sub_train_path, exist_ok=True)


ENT_WEIGHT= 1e-3
BATCH_SIZE= 512
LEARNING_RATE= 1e-4
NUMBER_TRAIN = 200
N_ENVS=4
N_STEPS= 1024 # 2048


env = SubprocVecEnv(
    [make_env(i, human=False) for i in range(N_ENVS)]
)

last_index_imitation = get_last_index(str(sub_train_path), "ppo_policy", "zip")

print(f"Last index ppo: {last_index_imitation}")

model = None

policy_kwargs = dict(
    lstm_hidden_size=512, 
    n_lstm_layers=1, 
    net_arch=dict(pi=[256, 256], vf=[256, 256]),
)

if last_index_imitation < 0:
    model = RecurrentPPO(
        policy="CnnLstmPolicy", 
        env=env,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        ent_coef=ENT_WEIGHT,
        n_steps=N_STEPS * N_ENVS,
        verbose=1,
        policy_kwargs=policy_kwargs,
        device="cuda"
    )
    
    print("Starting from zero model")
else:
    model = RecurrentPPO.load(
        f"{sub_train_path}ppo_policy{last_index_imitation}.zip", 
        env=env,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        ent_coef=ENT_WEIGHT,
        n_steps=N_STEPS * N_ENVS,
        verbose=1,
        policy_kwargs=policy_kwargs,
        device="cuda"
    )
    print(f"Loading model from ppo number: {last_index_imitation}")

model.set_logger(custom_logger)

current_epoch = last_index_imitation + 1

run_name = f"PPO_FineTuning_{current_time}"


for i in range(NUMBER_TRAIN):
    model.learn(
        total_timesteps=N_STEPS * N_ENVS * 2,
        reset_num_timesteps=False,
        tb_log_name=run_name
    )
    model.save(f"{sub_train_path}ppo_policy{current_epoch}.zip")

    current_epoch += 1
    
gc.collect()
del model

th.cuda.empty_cache()

print("Force cell kernel reset")
get_ipython().kernel.do_shutdown(restart=True)

D:\anaconda3\envs\env311\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Last index ppo: 160
Wrapping the env in a VecTransposeImage.


D:\anaconda3\envs\env311\Lib\site-packages\stable_baselines3\common\utils.py:166: UserWarning: get_schedule_fn() is deprecated, please use FloatSchedule() instead
  warnings.warn("get_schedule_fn() is deprecated, please use FloatSchedule() instead")


Loading model from ppo number: 160
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 455      |
|    ep_rew_mean     | 8.75     |
| time/              |          |
|    fps             | 43       |
|    iterations      | 1        |
|    time_elapsed    | 379      |
|    total_timesteps | 2949120  |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 357        |
|    ep_rew_mean          | 5.16       |
| time/                   |            |
|    fps                  | 43         |
|    iterations           | 1          |
|    time_elapsed         | 374        |
|    total_timesteps      | 2965504    |
| train/                  |            |
|    approx_kl            | 0.53467447 |
|    clip_fraction        | 0.239      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.215     |
|    explained_variance   | 0.889      |
|    learn